# 10 · Финальное обучение и submission

Шесть LightGBM для public и шесть для private. Параметры, деревья, веса и
калибровка **1.006** зафиксированы до этих запусков.

| Период | WRMSSE | Источник |
|---|---:|---|
| Историческое окно подбора | 0.489545 | Ноутбук 09 |
| Public: 25 апреля — 22 мая | 0.613284 | Сохранённый расчёт ниже |
| Private: 23 мая — 19 июня | **0.58009** | Поздняя отправка на Kaggle |

Private score получен поздней отправкой на Kaggle. SHA-256 файла:
`c5390842ef852d5400557249e1e24e24830fa280df3d75ba963d27a68996b66a`.

## 1. Запуск

Kaggle CPU, Input **M5 Forecasting — Accuracy** со всеми пятью файлами.
Выполнить **Save & Run All**. С нуля — 12 обучений; сохранённый запуск занял
**3 ч 52 мин** вместе с подготовкой и сохранением.

Для продолжения подключить `demand-planner_final_checkpoint.zip` из
[GitHub Release](https://github.com/dannya220/demand-planner/releases/tag/v1.0.0).
Готовые модели и прогнозы проходят проверку и пропускаются.

In [1]:
from pathlib import Path
import os
import sys

if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Обучение этого ноутбука запускаем на Kaggle CPU.")
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"

import gc
import gzip
import hashlib
import importlib.metadata
import io
import json
import platform
import shutil
import subprocess
import fcntl
from datetime import datetime, timezone
from functools import lru_cache
from time import perf_counter
from zipfile import ZipFile, ZIP_DEFLATED, is_zipfile

try:
    installed_lgbm = importlib.metadata.version("lightgbm")
except importlib.metadata.PackageNotFoundError:
    installed_lgbm = None
if installed_lgbm != "4.6.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "lightgbm==4.6.0"])

import lightgbm as lgb
import numpy as np
import pandas as pd
import scipy
from IPython.display import FileLink, display

if lgb.__version__ != "4.6.0":
    raise RuntimeError("Нужен новый kernel с LightGBM 4.6.0.")
pd.set_option("compute.use_bottleneck", False)
pd.set_option("display.max_columns", 25)
VERSIONS = {
    "python": platform.python_version(),
    "lightgbm": lgb.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
}
print("Среда:", VERSIONS)
print("Bottleneck:", pd.get_option("compute.use_bottleneck"))

Среда: {'python': '3.13.15', 'lightgbm': '4.6.0', 'numpy': '2.1.3', 'pandas': '2.3.3', 'scipy': '1.16.3'}
Bottleneck: False


## 2. Настройки запуска

`PERIODS_TO_RUN` определяет, какие периоды ещё обучать. Для полного submission
нужны оба. При повторном запуске готовые модели и прогнозы проверяются и пропускаются.

`TIMEOUT_HOURS` ограничивает одну попытку запуска. При нехватке времени сохраняются
завершённые модели; текущая незавершённая модель обучится заново после продолжения.
Данные и число деревьев от этого не меняются.

In [2]:
RUN_NAME = "final_m5_frozen_ensemble_v1"
PROJECT_ROOT = Path("/kaggle/working/demand-planner")
RESULT_DIR = PROJECT_ROOT / "results/m5_wrmsse" / RUN_NAME
MODEL_DIR = PROJECT_ROOT / "models" / RUN_NAME
MANIFEST_PATH = RESULT_DIR / "workspace_manifest.json"
CHECKPOINT_PATH = PROJECT_ROOT.parent / "demand-planner_final_checkpoint.zip"
SUBMISSION_PATH = PROJECT_ROOT.parent / "submission.csv"
RUN_TRAINING = True
PERIODS_TO_RUN = ["public", "private"]
TIMEOUT_HOURS = 11
NUM_THREADS = 4
RESUME_FROM = None

HORIZON = 28
N_ORIGINS = 52
M5_KEYS = ["store_id", "item_id"]
PUBLIC_ORIGIN = pd.Timestamp("2016-04-24")
PRIVATE_ORIGIN = pd.Timestamp("2016-05-22")
PERIOD_SPECS = {
    "public": {"origin": PUBLIC_ORIGIN, "suffix": "_validation", "last_day": 1913},
    "private": {"origin": PRIVATE_ORIGIN, "suffix": "_evaluation", "last_day": 1941},
}
if (
    not PERIODS_TO_RUN
    or len(set(PERIODS_TO_RUN)) != len(PERIODS_TO_RUN)
    or not set(PERIODS_TO_RUN).issubset(PERIOD_SPECS)
    or TIMEOUT_HOURS <= 0
    or NUM_THREADS < 1
):
    raise ValueError("Проверь периоды, время и число потоков.")

REQUIRED_FILES = (
    "calendar.csv",
    "sales_train_validation.csv",
    "sales_train_evaluation.csv",
    "sell_prices.csv",
    "sample_submission.csv",
)
data_candidates = sorted(
    {
        path.parent
        for path in Path("/kaggle/input").rglob("calendar.csv")
        if path.is_file() and all((path.parent / name).is_file() for name in REQUIRED_FILES)
    }
)
if len(data_candidates) != 1:
    raise FileNotFoundError(f"Подключи один полный набор M5: найдено {len(data_candidates)} папок.")
DATA_RAW = data_candidates[0]

## 3. Зафиксированный ансамбль моделей

Веса до калибровки:

| Неделя | Общая trial 9, 52 × 28 | Общая trial 5, 52 × 8 | Специалист недели |
|---|---:|---:|---:|
| 1 | 15% | 45% | 40% |
| 2 | 0% | 0% | 100% |
| 3 | 12.5% | 37.5% | 50% |
| 4 | 0% | 0% | 100% |

После смешивания весь прогноз умножается на **1.006**. Общая trial 5 на 52 × 28
получила нулевой вес и здесь не обучается. CatBoost, EWMA и отклонённые признаки
в финальный рецепт не входят.

Число деревьев: **197, 471** у общих моделей и **2076, 196, 216, 93** у недельных.
На public и private оно фиксировано: `early stopping` и Optuna здесь отсутствуют.

In [3]:
FROZEN_RECIPE = {
    "schema": 1,
    "name": "m5_frozen_horizon_ensemble_v1",
    "development_origin": "2016-03-27",
    "development_metrics": {"WRMSSE": 0.4895447561693653, "Bias": 0.0021079356482966993},
    "development_result": "ensemble_horizon_week1_3200_v1/best_blend/protocol.json",
    "development_protocol_sha256": "a3cb0ae87de0454d5e403e251a03e877936f80fb4efa3dd8a39ecb98384ac6bb",
    "calibration_coefficient": 1.006,
    "global_weights": {
        "global_trial9_28": 0.25,
        "global_trial5_28": 0.0,
        "global_trial5_8": 0.7499999999999999,
    },
    "specialist_weights_by_week": [0.4, 1.0, 0.5, 1.0],
    "features": [
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "mean_7d",
        "mean_28d",
        "mean_56d",
        "std_28d",
        "zero_share_28d",
        "trend_7d_28d",
        "weekday_mean_4w",
        "lag_28d",
        "lag_35d",
        "lag_42d",
        "lag_56d",
        "lag_364d",
        "horizon",
        "day_of_week",
        "day_of_month",
        "month",
        "week_of_year",
        "year",
        "snap",
        "has_event",
        "price_last_known",
        "price_observed_this_week",
        "price_change_1w",
        "price_change_4w",
        "price_vs_mean_4w",
        "price_vs_mean_13w",
        "price_vs_max_13w",
        "price_relative_std_13w",
        "price_observed_share_13w",
        "days_since_last_sale",
        "has_never_sold",
        "sale_day_share_7d",
        "sale_day_share_56d",
        "sale_frequency_change_7d_vs_prev21d",
        "mean_positive_sale_28d",
        "mean_positive_sale_56d",
        "positive_sale_cv2_56d",
        "max_daily_sale_28d",
        "store_dept_mean_7d",
        "store_dept_mean_28d",
        "store_dept_mean_56d",
        "store_dept_trend_7d_28d",
        "store_dept_trend_28d_56d",
        "store_dept_cv_28d",
        "store_dept_weekday_mean_4w",
        "store_dept_weekday_factor_4w",
        "item_share_store_dept_28d",
        "item_share_change_7d_28d",
        "other_stores_mean_7d",
        "other_stores_mean_28d",
        "other_stores_mean_56d",
        "other_stores_trend_7d_28d",
        "other_stores_trend_28d_56d",
        "other_stores_cv_28d",
        "other_stores_weekday_mean_4w",
        "other_stores_weekday_factor_4w",
        "own_vs_other_stores_mean_28d",
        "own_vs_other_stores_trend",
        "price_target_week",
        "price_target_vs_last",
        "price_target_vs_mean_13w",
        "price_target_missing",
    ],
    "categorical_features": ["item_id", "dept_id", "cat_id", "store_id", "state_id"],
    "feature_code_sha256": "9d8f183ca28c5935c8bb7341546a41ab54e1cd07df0863a93416007a142d6aeb",
    "models": [
        {
            "name": "global_trial9_28",
            "density": 28,
            "week": None,
            "trees": 197,
            "params": {
                "objective": "tweedie",
                "metric": "None",
                "boosting_type": "gbdt",
                "device_type": "cpu",
                "num_threads": 4,
                "seed": 42,
                "deterministic": True,
                "force_col_wise": True,
                "verbosity": -1,
                "max_depth": -1,
                "max_bin": 255,
                "bin_construct_sample_cnt": 200000,
                "feature_pre_filter": False,
                "bagging_freq": 1,
                "min_sum_hessian_in_leaf": 0.001,
                "min_gain_to_split": 0.0,
                "learning_rate": 0.09975125819860657,
                "num_leaves": 63,
                "min_data_in_leaf": 250,
                "lambda_l1": 0.8827248013286972,
                "lambda_l2": 1.740351379791848,
                "feature_fraction": 0.8563973660909573,
                "bagging_fraction": 0.7579657376635576,
                "tweedie_variance_power": 1.1026305531889093,
            },
        },
        {
            "name": "global_trial5_8",
            "density": 8,
            "week": None,
            "trees": 471,
            "params": {
                "objective": "tweedie",
                "metric": "None",
                "boosting_type": "gbdt",
                "device_type": "cpu",
                "num_threads": 4,
                "seed": 42,
                "deterministic": True,
                "force_col_wise": True,
                "verbosity": -1,
                "max_depth": -1,
                "max_bin": 255,
                "bin_construct_sample_cnt": 200000,
                "feature_pre_filter": False,
                "bagging_freq": 1,
                "min_sum_hessian_in_leaf": 0.001,
                "min_gain_to_split": 0.0,
                "learning_rate": 0.09579469687333796,
                "num_leaves": 63,
                "min_data_in_leaf": 250,
                "lambda_l1": 1.0682736700049809,
                "lambda_l2": 0.11322701075434688,
                "feature_fraction": 0.997966521134667,
                "bagging_fraction": 0.7128285798547571,
                "tweedie_variance_power": 1.1168250742845671,
            },
        },
        {
            "name": "week_01",
            "density": 28,
            "week": 1,
            "trees": 2076,
            "params": {
                "objective": "tweedie",
                "metric": "None",
                "boosting_type": "gbdt",
                "device_type": "cpu",
                "num_threads": 4,
                "seed": 42,
                "deterministic": True,
                "force_col_wise": True,
                "verbosity": -1,
                "max_depth": -1,
                "max_bin": 255,
                "bin_construct_sample_cnt": 200000,
                "feature_pre_filter": False,
                "bagging_freq": 1,
                "min_sum_hessian_in_leaf": 0.001,
                "min_gain_to_split": 0.0,
                "learning_rate": 0.09975125819860657,
                "num_leaves": 63,
                "min_data_in_leaf": 250,
                "lambda_l1": 0.8827248013286972,
                "lambda_l2": 1.740351379791848,
                "feature_fraction": 0.8563973660909573,
                "bagging_fraction": 0.7579657376635576,
                "tweedie_variance_power": 1.1026305531889093,
            },
        },
        {
            "name": "week_02",
            "density": 28,
            "week": 2,
            "trees": 196,
            "params": {
                "objective": "tweedie",
                "metric": "None",
                "boosting_type": "gbdt",
                "device_type": "cpu",
                "num_threads": 4,
                "seed": 42,
                "deterministic": True,
                "force_col_wise": True,
                "verbosity": -1,
                "max_depth": -1,
                "max_bin": 255,
                "bin_construct_sample_cnt": 200000,
                "feature_pre_filter": False,
                "bagging_freq": 1,
                "min_sum_hessian_in_leaf": 0.001,
                "min_gain_to_split": 0.0,
                "learning_rate": 0.09975125819860657,
                "num_leaves": 63,
                "min_data_in_leaf": 250,
                "lambda_l1": 0.8827248013286972,
                "lambda_l2": 1.740351379791848,
                "feature_fraction": 0.8563973660909573,
                "bagging_fraction": 0.7579657376635576,
                "tweedie_variance_power": 1.1026305531889093,
            },
        },
        {
            "name": "week_03",
            "density": 28,
            "week": 3,
            "trees": 216,
            "params": {
                "objective": "tweedie",
                "metric": "None",
                "boosting_type": "gbdt",
                "device_type": "cpu",
                "num_threads": 4,
                "seed": 42,
                "deterministic": True,
                "force_col_wise": True,
                "verbosity": -1,
                "max_depth": -1,
                "max_bin": 255,
                "bin_construct_sample_cnt": 200000,
                "feature_pre_filter": False,
                "bagging_freq": 1,
                "min_sum_hessian_in_leaf": 0.001,
                "min_gain_to_split": 0.0,
                "learning_rate": 0.09975125819860657,
                "num_leaves": 63,
                "min_data_in_leaf": 250,
                "lambda_l1": 0.8827248013286972,
                "lambda_l2": 1.740351379791848,
                "feature_fraction": 0.8563973660909573,
                "bagging_fraction": 0.7579657376635576,
                "tweedie_variance_power": 1.1026305531889093,
            },
        },
        {
            "name": "week_04",
            "density": 28,
            "week": 4,
            "trees": 93,
            "params": {
                "objective": "tweedie",
                "metric": "None",
                "boosting_type": "gbdt",
                "device_type": "cpu",
                "num_threads": 4,
                "seed": 42,
                "deterministic": True,
                "force_col_wise": True,
                "verbosity": -1,
                "max_depth": -1,
                "max_bin": 255,
                "bin_construct_sample_cnt": 200000,
                "feature_pre_filter": False,
                "bagging_freq": 1,
                "min_sum_hessian_in_leaf": 0.001,
                "min_gain_to_split": 0.0,
                "learning_rate": 0.09975125819860657,
                "num_leaves": 63,
                "min_data_in_leaf": 250,
                "lambda_l1": 0.8827248013286972,
                "lambda_l2": 1.740351379791848,
                "feature_fraction": 0.8563973660909573,
                "bagging_fraction": 0.7579657376635576,
                "tweedie_variance_power": 1.1026305531889093,
            },
        },
    ],
    "tree_selection": "fixed_before_public_and_private_from_historical_development",
    "label_sampling": "52_weekly_origins_with_the_original_8_or_28_horizon_plan",
}
FEATURE_COLUMNS = FROZEN_RECIPE["features"]
CATEGORICAL_FEATURES = FROZEN_RECIPE["categorical_features"]
BASE_FEATURE_COLUMNS = FEATURE_COLUMNS[:-4]
MODEL_SPECS = FROZEN_RECIPE["models"]
CALIBRATION_COEFFICIENT = FROZEN_RECIPE["calibration_coefficient"]


def component_weights_by_week(recipe):
    names = [spec["name"] for spec in recipe["models"]]
    weights = pd.DataFrame(0.0, index=pd.Index(range(1, 5), name="week"), columns=names)
    for week, alpha in enumerate(recipe["specialist_weights_by_week"], start=1):
        if not 0 <= alpha <= 1:
            raise ValueError("Неверный вес специалиста.")
        for name, weight in recipe["global_weights"].items():
            if name not in weights:
                if weight != 0:
                    raise ValueError(f"Нет ненулевой модели {name}.")
                continue
            weights.loc[week, name] = (1 - alpha) * weight
        weights.loc[week, f"week_{week:02d}"] = alpha
    if (weights.to_numpy() < 0).any() or not np.allclose(weights.sum(axis=1), 1):
        raise ValueError("Веса должны быть неотрицательными и суммироваться в 1.")
    return weights


WEEK_WEIGHTS = component_weights_by_week(FROZEN_RECIPE)
if len(FEATURE_COLUMNS) != 67 or len(MODEL_SPECS) != 6:
    raise ValueError("Изменился выбранный рецепт.")
display(
    pd.DataFrame(
        [{key: spec[key] for key in ("name", "trees", "density", "week")} for spec in MODEL_SPECS]
    ).set_index("name")
)
display(WEEK_WEIGHTS.style.format("{:.1%}"))
print("Калибровка:", CALIBRATION_COEFFICIENT)

,trees,density,week
name,,,
global_trial9_28,197,28,NaN
global_trial5_8,471,8,NaN
week_01,2076,28,1.0
week_02,196,28,2.0
week_03,216,28,3.0
week_04,93,28,4.0


,global_trial9_28,global_trial5_8,week_01,week_02,week_03,week_04
week,,,,,,
1,15.0%,45.0%,40.0%,0.0%,0.0%,0.0%
2,0.0%,0.0%,0.0%,100.0%,0.0%,0.0%
3,12.5%,37.5%,0.0%,0.0%,50.0%,0.0%
4,0.0%,0.0%,0.0%,0.0%,0.0%,100.0%


Калибровка: 1.006


## 4. Данные и границы истории

Из `sales_train_evaluation.csv` читаем известные продажи **d_1–d_1941**.
Перед обучением public история физически обрезается по **d_1913**. Факты следующих
28 дней используются только после обучения для отчёта о качестве.

Перед private разрешены все **d_1–d_1941**. Будущих продаж d_1942–d_1969 в данных нет.
Календарь и цены целевой недели доступны для обоих периодов по условиям M5.

Вся доступная история участвует в исторических признаках. Обучающие строки
сохраняют проверенную сетку **52 даты отсчёта**: 28 горизонтов для общей trial 9,
8 для общей trial 5 и семь нужных горизонтов для каждого специалиста. Увеличивать
этот план на финальном шаге без проверки не будем.

In [4]:
def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
    )
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as incoming:
        for block in iter(lambda: incoming.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def object_hash(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()
    ).hexdigest()


def read_csv_payload(path):
    data = Path(path).read_bytes()
    return gzip.decompress(data) if data.startswith(bytes((31, 139))) else data


def resolve_prediction_payload(directory, expected_hash):
    directory = Path(directory)
    candidates = []
    for name in ("predictions.csv.gz", "predictions.csv"):
        path = directory / name
        if path.is_file():
            candidates.append(path)
        elif path.is_dir():
            candidates.extend(p for p in path.rglob("*") if p.is_file())
    for path in sorted(set(candidates)):
        payload = read_csv_payload(path)
        if hashlib.sha256(payload).hexdigest() == expected_hash:
            return path, payload
    raise RuntimeError(f"Не найден прогноз с верным хешем: {directory}")


print("Проверяю исходные CSV.")
RAW_SHA256 = {name: sha256_file(DATA_RAW / name) for name in REQUIRED_FILES}
EXPECTED_SOURCE_HASHES = {
    "calendar.csv": "d12b5914ef03e66649adf5dd9e996e6602251c22b7a6af8f1f7e3aa12f8860f5",
    "sales_train_validation.csv": "f368e66ed1dbecb48b2cc8fc589bf68b3deddbbb36bf5c88b4d6d0a09b9b6724",
    "sell_prices.csv": "9da3ad1f8b8ccacdbdc70612191dd375ec24a4ac6625c24b75b3bc60b0bed2ef",
}
if any(RAW_SHA256[name] != digest for name, digest in EXPECTED_SOURCE_HASHES.items()):
    raise RuntimeError("Исходные файлы отличаются от использованных в экспериментах.")

Проверяю исходные CSV.


In [5]:
calendar = pd.read_csv(DATA_RAW / "calendar.csv", parse_dates=["date"])
if calendar["date"].duplicated().any() or calendar["d"].duplicated().any():
    raise ValueError("Повторы дат в календаре.")
daily_calendar = calendar.set_index("date")
identifier_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
series_metadata = pd.read_csv(
    DATA_RAW / "sales_train_validation.csv", usecols=identifier_columns, dtype="string"
)
evaluation_metadata = pd.read_csv(
    DATA_RAW / "sales_train_evaluation.csv", usecols=identifier_columns, dtype="string"
)
series_index = pd.MultiIndex.from_frame(series_metadata[M5_KEYS])
if len(series_index) != 30_490 or series_index.has_duplicates:
    raise ValueError("Нужны все 30 490 рядов M5 без повторов.")
aligned_evaluation = evaluation_metadata.set_index(M5_KEYS).reindex(series_index)
if aligned_evaluation.isna().any().any():
    raise ValueError("Не совпали ряды validation и evaluation.")
for column in ("dept_id", "cat_id", "state_id"):
    if not np.array_equal(
        aligned_evaluation[column].to_numpy(), series_metadata[column].to_numpy()
    ):
        raise ValueError(f"Не совпала metadata: {column}.")
validation_ids = pd.Index(series_metadata["id"].astype(str), name="id")
evaluation_ids = pd.Index(aligned_evaluation["id"].astype(str), name="id")
if (
    not validation_ids.str.endswith("_validation").all()
    or not evaluation_ids.str.endswith("_evaluation").all()
    or not np.array_equal(
        validation_ids.str.removesuffix("_validation"),
        evaluation_ids.str.removesuffix("_evaluation"),
    )
):
    raise ValueError("Неверные суффиксы или порядок конкурсных id.")
SUBMISSION_IDS = {"public": validation_ids, "private": evaluation_ids}

train_day_columns = [f"d_{day}" for day in range(1, 1942)]
train_daily = (
    pd.read_csv(
        DATA_RAW / "sales_train_evaluation.csv",
        usecols=[*M5_KEYS, *train_day_columns],
        dtype={
            **{key: "string" for key in M5_KEYS},
            **{column: "int32" for column in train_day_columns},
        },
    )
    .set_index(M5_KEYS)
    .reindex(series_index)
    .loc[:, train_day_columns]
)
train_daily.columns = pd.DatetimeIndex(
    calendar.set_index("d").loc[train_day_columns, "date"], name="date"
)
if (
    train_daily.columns[-1] != PRIVATE_ORIGIN
    or not np.isfinite(train_daily.to_numpy()).all()
    or (train_daily.to_numpy() < 0).any()
):
    raise ValueError("Неверный конец истории или некорректные продажи.")

validation_columns = train_day_columns[:1913]
validation_sales = (
    pd.read_csv(
        DATA_RAW / "sales_train_validation.csv",
        usecols=[*M5_KEYS, *validation_columns],
        dtype={
            **{key: "string" for key in M5_KEYS},
            **{column: "int32" for column in validation_columns},
        },
    )
    .set_index(M5_KEYS)
    .reindex(series_index)
    .loc[:, validation_columns]
)
if not np.array_equal(validation_sales.to_numpy(), train_daily.iloc[:, :1913].to_numpy()):
    raise ValueError("Продажи d_1–d_1913 в двух файлах различаются.")
del validation_sales
sample_submission = pd.read_csv(DATA_RAW / "sample_submission.csv", dtype={"id": "string"})

M5_LEVELS = {
    "Total": [],
    "State": ["state_id"],
    "Store": ["store_id"],
    "Category": ["cat_id"],
    "Department": ["dept_id"],
    "State / Category": ["state_id", "cat_id"],
    "State / Department": ["state_id", "dept_id"],
    "Store / Category": ["store_id", "cat_id"],
    "Store / Department": ["store_id", "dept_id"],
    "Item": ["item_id"],
    "Item / State": ["item_id", "state_id"],
    "Item / Store": ["item_id", "store_id"],
}
weight_dates = train_daily.loc[:, :PUBLIC_ORIGIN].columns[-28:]
weight_weeks = daily_calendar.loc[weight_dates, "wm_yr_wk"].unique()
price_parts = []
for chunk in pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    dtype={"store_id": "string", "item_id": "string", "wm_yr_wk": "int32", "sell_price": "float64"},
    chunksize=200_000,
):
    part = chunk.loc[chunk["wm_yr_wk"].isin(weight_weeks)]
    if not part.empty:
        price_parts.append(part.copy())
prices_for_weights = pd.concat(price_parts, ignore_index=True)
del price_parts, chunk, part
gc.collect()
print("Рядов:", len(train_daily), "| Известных дней:", train_daily.shape[1])

Рядов: 30490 | Известных дней: 1941


In [6]:
from scipy.sparse import csr_matrix


def calculate_rmsse_scale(history_values, aggregation, block_size=128):
    n_groups = aggregation.shape[0]
    n_days = history_values.shape[1]

    first_nonzero = np.full(n_groups, n_days, dtype=np.int32)
    squared_difference_sum = np.zeros(n_groups, dtype=np.float64)
    previous = np.zeros(n_groups, dtype=np.float64)

    for start in range(0, n_days, block_size):
        stop = min(start + block_size, n_days)

        block = np.asarray(
            aggregation @ history_values[:, start:stop],
            dtype=np.float64,
        )

        positive = block > 0
        newly_active = (first_nonzero == n_days) & positive.any(axis=1)

        if newly_active.any():
            first_nonzero[newly_active] = start + positive[newly_active].argmax(axis=1)

        differences = np.empty_like(block)
        differences[:, 0] = block[:, 0] - previous
        differences[:, 1:] = np.diff(block, axis=1)

        positions = np.arange(start, stop)
        exclude = positions[None, :] <= first_nonzero[:, None]
        differences[exclude] = 0.0

        squared_difference_sum += np.einsum(
            "ij,ij->i",
            differences,
            differences,
        )

        previous = block[:, -1].copy()

    difference_count = n_days - first_nonzero - 1

    return np.divide(
        squared_difference_sum,
        difference_count,
        out=np.zeros(n_groups, dtype=np.float64),
        where=difference_count > 0,
    )


class M5WRMSSE:
    def __init__(
        self,
        history,
        metadata,
        calendar,
        prices,
        levels,
        horizon=28,
    ):
        if len(metadata) != 30_490 or len(levels) != 12:
            raise ValueError("Для полного M5 нужны 30 490 рядов и 12 уровней.")

        self.series_index = pd.MultiIndex.from_frame(metadata[M5_KEYS])

        if (
            self.series_index.has_duplicates
            or history.index.has_duplicates
            or len(history) != len(self.series_index)
            or not self.series_index.isin(history.index).all()
        ):
            raise ValueError("Идентификаторы истории и metadata не совпадают.")

        history = history.reindex(self.series_index)

        if history.shape[1] < 28:
            raise ValueError("Для расчёта весов нужны минимум 28 дней.")

        if not isinstance(history.columns, pd.DatetimeIndex):
            raise TypeError("Столбцы истории должны содержать даты.")

        expected_history_dates = pd.date_range(
            history.columns[0],
            periods=history.shape[1],
            freq="D",
        )

        if not history.columns.equals(expected_history_dates):
            raise ValueError("В истории есть пропуски или нарушен порядок дней.")

        history_values = history.to_numpy(copy=False)

        if not np.isfinite(history_values).all() or (history_values < 0).any():
            raise ValueError("История должна содержать конечные неотрицательные продажи.")

        self.horizon = horizon
        self.n_levels = len(levels)
        self.forecast_dates = pd.date_range(
            history.columns[-1] + pd.Timedelta(days=1),
            periods=horizon,
            freq="D",
            name="date",
        )

        recent_dates = history.columns[-28:]

        week_codes = calendar.set_index("date").reindex(recent_dates)["wm_yr_wk"].to_numpy()

        if pd.isna(week_codes).any():
            raise ValueError("В календаре отсутствуют даты для расчёта весов.")

        known_prices = prices["sell_price"].dropna().to_numpy()

        if not np.isfinite(known_prices).all() or (known_prices <= 0).any():
            raise ValueError("Известные цены должны быть конечными и положительными.")

        price_table = prices.pivot(
            index=M5_KEYS,
            columns="wm_yr_wk",
            values="sell_price",
        )

        daily_prices = price_table.reindex(
            index=self.series_index,
            columns=week_codes,
        ).to_numpy(dtype=np.float64)

        recent_units = history.iloc[:, -28:].to_numpy(dtype=np.float64)

        missing_price_with_sales = (recent_units > 0) & ~np.isfinite(daily_prices)

        if missing_price_with_sales.any():
            raise ValueError(
                "Есть положительные продажи без цены: веса нельзя посчитать корректно."
            )

        dollar_sales = (recent_units * np.nan_to_num(daily_prices, nan=0.0)).sum(axis=1)

        total_dollar_sales = dollar_sales.sum()

        if total_dollar_sales <= 0:
            raise ValueError("Суммарные денежные продажи равны нулю.")

        metadata = metadata.reset_index(drop=True)

        self.level_specs = {}
        level_rows = []

        for level_name, columns in levels.items():
            if columns:
                group_index = pd.MultiIndex.from_frame(metadata[columns])
                codes, groups = pd.factorize(
                    group_index,
                    sort=True,
                )
                n_groups = len(groups)
            else:
                codes = np.zeros(len(metadata), dtype=np.int32)
                n_groups = 1

            aggregation = csr_matrix(
                (
                    np.ones(len(metadata), dtype=np.float64),
                    (codes, np.arange(len(metadata))),
                ),
                shape=(n_groups, len(metadata)),
            )

            print(f"Масштабы: {level_name} — {n_groups:,} рядов")

            scale = calculate_rmsse_scale(
                history_values,
                aggregation,
            )

            group_dollar_sales = np.asarray(aggregation @ dollar_sales).ravel()

            weights = group_dollar_sales / total_dollar_sales / self.n_levels

            if ((scale <= 0) & (weights > 0)).any():
                raise ValueError(
                    f"{level_name}: есть ряд с положительным весом " "и нулевым масштабом RMSSE."
                )

            self.level_specs[level_name] = {
                "aggregation": aggregation,
                "scale": scale,
                "weights": weights,
            }

            level_rows.append(
                {
                    "level": level_name,
                    "series": n_groups,
                    "weight_sum": weights.sum(),
                }
            )

        self.level_info = pd.DataFrame(level_rows).set_index("level")

        if self.level_info["series"].sum() != 42_840:
            raise ValueError("Получено неверное число рядов иерархии.")

        if not np.isclose(self.level_info["weight_sum"].sum(), 1.0):
            raise ValueError("Сумма весов WRMSSE отличается от единицы.")

    def _aligned_values(self, frame):
        if not isinstance(frame, pd.DataFrame):
            raise TypeError("Передай DataFrame с идентификаторами и датами.")

        if (
            frame.index.has_duplicates
            or len(frame) != len(self.series_index)
            or not self.series_index.isin(frame.index).all()
        ):
            raise ValueError("Набор товаров и магазинов не совпадает.")

        if not frame.columns.equals(self.forecast_dates):
            raise ValueError("Столбцы должны содержать следующие 28 дат " "в правильном порядке.")

        values = frame.reindex(self.series_index).to_numpy(dtype=np.float64)

        if not np.isfinite(values).all():
            raise ValueError("В фактах или прогнозах есть NaN либо бесконечность.")

        return values

    def score(self, actual, prediction):
        actual_values = self._aligned_values(actual)
        prediction_values = self._aligned_values(prediction)

        bottom_errors = prediction_values - actual_values
        rows = []

        for level_name, spec in self.level_specs.items():
            aggregated_errors = spec["aggregation"] @ bottom_errors

            mse = np.mean(
                np.square(aggregated_errors),
                axis=1,
            )

            rmsse = np.sqrt(
                np.divide(
                    mse,
                    spec["scale"],
                    out=np.zeros_like(mse),
                    where=spec["scale"] > 0,
                )
            )

            contribution = float(np.dot(spec["weights"], rmsse))

            rows.append(
                {
                    "level": level_name,
                    "weighted_RMSSE": contribution * self.n_levels,
                    "contribution": contribution,
                }
            )

        report = pd.DataFrame(rows).set_index("level")
        overall = float(report["contribution"].sum())

        return overall, report

In [7]:
wrmsse_evaluator = M5WRMSSE(
    history=train_daily.loc[:, :PUBLIC_ORIGIN],
    metadata=series_metadata,
    calendar=calendar,
    prices=prices_for_weights,
    levels=M5_LEVELS,
    horizon=HORIZON,
)
if wrmsse_evaluator.forecast_dates[-1] != PRIVATE_ORIGIN:
    raise ValueError("Не совпал public-период.")
cb_public_origin = PRIVATE_ORIGIN

Масштабы: Total — 1 рядов
Масштабы: State — 3 рядов
Масштабы: Store — 10 рядов
Масштабы: Category — 3 рядов
Масштабы: Department — 7 рядов
Масштабы: State / Category — 9 рядов
Масштабы: State / Department — 21 рядов
Масштабы: Store / Category — 30 рядов
Масштабы: Store / Department — 70 рядов
Масштабы: Item — 3,049 рядов
Масштабы: Item / State — 9,147 рядов
Масштабы: Item / Store — 30,490 рядов


## 5. Признаки

Те же **67 признаков**: история продаж, цены, календарь и SNAP, регулярность
покупок, динамика отдела, история товара в других магазинах и цена целевой недели.
Функции перенесены из ноутбука 08 без изменения расчётов. Bottleneck выключен
до чтения и обработки таблиц.

Продажи для признаков обрезаются по дате отсчёта. Цена целевой недели и календарь
могут относиться к будущему: эти данные разрешены и доступны в M5.

In [8]:
daily_categorical_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

daily_history = train_daily.reindex(wrmsse_evaluator.series_index)

daily_history_dates = daily_history.columns
daily_history_values = daily_history.to_numpy(copy=False)

daily_identifiers = (
    series_metadata.set_index(M5_KEYS)
    .loc[wrmsse_evaluator.series_index]
    .reset_index()
    .loc[:, daily_categorical_features]
)

daily_identifiers.index = wrmsse_evaluator.series_index

for column in daily_categorical_features:
    daily_identifiers[column] = daily_identifiers[column].astype("category")

daily_calendar = calendar.set_index("date")


def make_daily_features(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)

    horizon = (target_date - forecast_origin).days

    if not 1 <= horizon <= HORIZON:
        raise ValueError("Горизонт должен быть от 1 до 28 дней.")

    origin_position = daily_history_dates.get_indexer([forecast_origin])[0]

    if origin_position < 55:
        raise ValueError(
            "Дата прогноза должна присутствовать в истории " "и иметь минимум 56 доступных дней."
        )

    X = daily_identifiers.copy()
    known_history = daily_history_values[:, : origin_position + 1]

    for window in (7, 28, 56):
        X[f"mean_{window}d"] = known_history[:, -window:].mean(axis=1).astype("float32")

    recent_28 = known_history[:, -28:]

    X["std_28d"] = recent_28.std(axis=1, ddof=1).astype("float32")

    X["zero_share_28d"] = (recent_28 == 0).mean(axis=1).astype("float32")

    X["trend_7d_28d"] = (X["mean_7d"] - X["mean_28d"]).astype("float32")

    recent_dates = daily_history_dates[origin_position - 27 : origin_position + 1]

    weekday_mask = recent_dates.dayofweek == target_date.dayofweek

    X["weekday_mean_4w"] = recent_28[:, weekday_mask].mean(axis=1).astype("float32")

    for lag in (28, 35, 42, 56, 364):
        source_date = target_date - pd.Timedelta(days=lag)

        if source_date > forecast_origin:
            raise ValueError("Лаг обращается к неизвестному будущему.")

        source_position = daily_history_dates.get_indexer([source_date])[0]

        if source_position >= 0:
            X[f"lag_{lag}d"] = daily_history_values[:, source_position].astype("float32")
        else:
            X[f"lag_{lag}d"] = np.float32(np.nan)

    calendar_values = daily_calendar.loc[target_date]

    target_features = {
        "horizon": horizon,
        "day_of_week": target_date.dayofweek,
        "day_of_month": target_date.day,
        "month": target_date.month,
        "week_of_year": int(target_date.isocalendar().week),
        "year": target_date.year,
    }

    for name, value in target_features.items():
        X[name] = np.int16(value)

    snap_by_state = {state: int(calendar_values[f"snap_{state}"]) for state in ("CA", "TX", "WI")}

    X["snap"] = X["state_id"].map(snap_by_state).astype("int8")

    X["has_event"] = np.int8(
        pd.notna(calendar_values["event_name_1"]) or pd.notna(calendar_values["event_name_2"])
    )

    return X

In [9]:
price_data = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    dtype={
        "store_id": "category",
        "item_id": "category",
        "wm_yr_wk": "int32",
        "sell_price": "float64",
    },
)
price_week_starts = calendar.groupby("wm_yr_wk")["date"].min().sort_values()
price_week_order = price_week_starts.loc[price_week_starts <= cb_public_origin].index
price_data = price_data.loc[price_data["wm_yr_wk"].isin(price_week_order)].copy()
price_week_table = (
    price_data.pivot(index=M5_KEYS, columns="wm_yr_wk", values="sell_price")
    .reindex(index=wrmsse_evaluator.series_index, columns=price_week_order)
    .astype("float32")
)
price_last_known_table = price_week_table.ffill(axis=1)


@lru_cache(maxsize=64)
def price_features_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    week = int(daily_calendar.loc[forecast_origin, "wm_yr_wk"])
    position = price_week_table.columns.get_loc(week)
    if position < 12:
        raise ValueError("Для признаков цены нужны 13 недель истории.")
    recent_4 = price_week_table.iloc[:, position - 3 : position + 1]
    recent_13 = price_week_table.iloc[:, position - 12 : position + 1]
    current = price_last_known_table.iloc[:, position]
    previous_1 = price_last_known_table.iloc[:, position - 1]
    previous_4 = price_last_known_table.iloc[:, position - 4]
    mean_4 = recent_4.mean(axis=1)
    mean_13 = recent_13.mean(axis=1)
    return pd.DataFrame(
        {
            "price_last_known": current,
            "price_observed_this_week": price_week_table.iloc[:, position].notna(),
            "price_change_1w": current / previous_1 - 1,
            "price_change_4w": current / previous_4 - 1,
            "price_vs_mean_4w": current / mean_4 - 1,
            "price_vs_mean_13w": current / mean_13 - 1,
            "price_vs_max_13w": current / recent_13.max(axis=1) - 1,
            "price_relative_std_13w": recent_13.std(axis=1, ddof=0) / mean_13,
            "price_observed_share_13w": recent_13.notna().mean(axis=1),
        }
    ).astype("float32")


def make_daily_features_with_price(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    base = make_daily_features(forecast_origin=forecast_origin, target_date=target_date)
    return base.join(price_features_at_origin(forecast_origin), validate="one_to_one")

In [10]:
@lru_cache(maxsize=64)
def regularity_features_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Дата должна присутствовать в истории и иметь минимум 56 доступных дней.")
    known = daily_history_values[:, : position + 1]
    positive_history = known > 0
    has_ever_sold = positive_history.any(axis=1)
    days_since_sale = positive_history[:, ::-1].argmax(axis=1).astype("float32")
    days_since_sale[~has_ever_sold] = np.nan
    recent_56 = known[:, -56:].astype("float64")
    recent_28 = recent_56[:, -28:]
    positive_56 = positive_history[:, -56:]
    positive_28 = positive_56[:, -28:]
    count_56 = positive_56.sum(axis=1)
    count_28 = positive_28.sum(axis=1)
    sum_56 = recent_56.sum(axis=1)
    sum_28 = recent_28.sum(axis=1)
    mean_positive_56 = np.divide(
        sum_56, count_56, out=np.full(len(known), np.nan), where=count_56 > 0
    )
    mean_positive_28 = np.divide(
        sum_28, count_28, out=np.full(len(known), np.nan), where=count_28 > 0
    )
    squared_sum_56 = np.einsum("ij,ij->i", recent_56, recent_56)
    correction = np.divide(
        np.square(sum_56), count_56, out=np.zeros(len(known)), where=count_56 > 0
    )
    positive_variance_56 = np.divide(
        np.maximum(squared_sum_56 - correction, 0.0),
        count_56 - 1,
        out=np.full(len(known), np.nan),
        where=count_56 > 1,
    )
    positive_cv2_56 = np.divide(
        positive_variance_56,
        np.square(mean_positive_56),
        out=np.full(len(known), np.nan),
        where=count_56 > 1,
    )
    frequency_7 = positive_56[:, -7:].mean(axis=1)
    frequency_previous_21 = positive_56[:, -28:-7].mean(axis=1)
    return pd.DataFrame(
        {
            "days_since_last_sale": days_since_sale,
            "has_never_sold": ~has_ever_sold,
            "sale_day_share_7d": frequency_7,
            "sale_day_share_56d": positive_56.mean(axis=1),
            "sale_frequency_change_7d_vs_prev21d": frequency_7 - frequency_previous_21,
            "mean_positive_sale_28d": mean_positive_28,
            "mean_positive_sale_56d": mean_positive_56,
            "positive_sale_cv2_56d": positive_cv2_56,
            "max_daily_sale_28d": recent_28.max(axis=1),
        },
        index=wrmsse_evaluator.series_index,
    ).astype("float32")


def make_daily_features_with_regularity(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    X = make_daily_features_with_price(forecast_origin, target_date)
    return X.join(regularity_features_at_origin(forecast_origin), validate="one_to_one")

In [11]:
group_aggregation = wrmsse_evaluator.level_specs["Store / Department"]["aggregation"]
group_membership = group_aggregation.tocoo()
group_codes = np.empty(len(wrmsse_evaluator.series_index), dtype=np.int32)
group_codes[group_membership.col] = group_membership.row
group_daily_sales = np.asarray(group_aggregation @ daily_history_values, dtype=np.float64)


def group_safe_ratio(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    return np.divide(
        numerator,
        denominator,
        out=np.full(numerator.shape, np.nan),
        where=denominator > 0,
    )


@lru_cache(maxsize=64)
def group_statistics_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Для групповых признаков нужны 56 доступных дней.")
    known = group_daily_sales[:, : position + 1]
    recent_28 = known[:, -28:]
    recent_dates = daily_history_dates[position - 27 : position + 1]
    statistics = {f"mean_{window}d": known[:, -window:].mean(axis=1) for window in (7, 28, 56)}
    statistics["std_28d"] = recent_28.std(axis=1, ddof=1)
    statistics["weekday_mean_4w"] = np.column_stack(
        [recent_28[:, recent_dates.dayofweek == weekday].mean(axis=1) for weekday in range(7)]
    )
    return statistics


def make_daily_features_with_groups(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    X = make_daily_features_with_regularity(forecast_origin, target_date)
    statistics = group_statistics_at_origin(forecast_origin)
    mean_7 = statistics["mean_7d"][group_codes]
    mean_28 = statistics["mean_28d"][group_codes]
    mean_56 = statistics["mean_56d"][group_codes]
    weekday_mean = statistics["weekday_mean_4w"][group_codes, target_date.dayofweek]
    item_share_7 = group_safe_ratio(X["mean_7d"].to_numpy(), mean_7)
    item_share_28 = group_safe_ratio(X["mean_28d"].to_numpy(), mean_28)
    added_features = {
        "store_dept_mean_7d": mean_7,
        "store_dept_mean_28d": mean_28,
        "store_dept_mean_56d": mean_56,
        "store_dept_trend_7d_28d": group_safe_ratio(mean_7, mean_28) - 1,
        "store_dept_trend_28d_56d": group_safe_ratio(mean_28, mean_56) - 1,
        "store_dept_cv_28d": group_safe_ratio(statistics["std_28d"][group_codes], mean_28),
        "store_dept_weekday_mean_4w": weekday_mean,
        "store_dept_weekday_factor_4w": group_safe_ratio(weekday_mean, mean_28),
        "item_share_store_dept_28d": item_share_28,
        "item_share_change_7d_28d": item_share_7 - item_share_28,
    }
    for column, values in added_features.items():
        X[column] = values.astype("float32")
    return X

In [12]:
cross_store_aggregation = wrmsse_evaluator.level_specs["Item"]["aggregation"]
cross_store_membership = cross_store_aggregation.tocoo()
cross_store_codes = np.empty(len(wrmsse_evaluator.series_index), dtype=np.int32)
cross_store_codes[cross_store_membership.col] = cross_store_membership.row
cross_store_counts = np.asarray(cross_store_aggregation.sum(axis=1)).ravel()
cross_store_other_counts = cross_store_counts[cross_store_codes] - 1
cross_store_daily_totals = np.asarray(
    cross_store_aggregation @ daily_history_values, dtype=np.float64
)


@lru_cache(maxsize=64)
def cross_store_statistics_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Для признаков других магазинов нужны 56 дней истории.")
    start = position - 55
    network_sales = cross_store_daily_totals[cross_store_codes, start : position + 1]
    own_sales = daily_history_values[:, start : position + 1]
    other_sales = (network_sales - own_sales) / cross_store_other_counts[:, None]
    recent_28 = other_sales[:, -28:]
    recent_dates = daily_history_dates[position - 27 : position + 1]
    statistics = {
        f"mean_{window}d": other_sales[:, -window:].mean(axis=1).astype("float32")
        for window in (7, 28, 56)
    }
    statistics["std_28d"] = recent_28.std(axis=1, ddof=1).astype("float32")
    statistics["weekday_mean_4w"] = np.column_stack(
        [recent_28[:, recent_dates.dayofweek == weekday].mean(axis=1) for weekday in range(7)]
    ).astype("float32")
    return statistics


def make_daily_features_with_cross_store(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    X = make_daily_features_with_groups(forecast_origin, target_date)
    statistics = cross_store_statistics_at_origin(forecast_origin)
    mean_7 = statistics["mean_7d"]
    mean_28 = statistics["mean_28d"]
    mean_56 = statistics["mean_56d"]
    weekday_mean = statistics["weekday_mean_4w"][:, target_date.dayofweek]
    own_trend = group_safe_ratio(X["mean_7d"].to_numpy(), X["mean_28d"].to_numpy()) - 1
    other_trend = group_safe_ratio(mean_7, mean_28) - 1
    added_features = {
        "other_stores_mean_7d": mean_7,
        "other_stores_mean_28d": mean_28,
        "other_stores_mean_56d": mean_56,
        "other_stores_trend_7d_28d": other_trend,
        "other_stores_trend_28d_56d": group_safe_ratio(mean_28, mean_56) - 1,
        "other_stores_cv_28d": group_safe_ratio(statistics["std_28d"], mean_28),
        "other_stores_weekday_mean_4w": weekday_mean,
        "other_stores_weekday_factor_4w": group_safe_ratio(weekday_mean, mean_28),
        "own_vs_other_stores_mean_28d": group_safe_ratio(X["mean_28d"].to_numpy(), mean_28),
        "own_vs_other_stores_trend": own_trend - other_trend,
    }
    for column, values in added_features.items():
        X[column] = values.astype("float32")
    return X

In [13]:
CATEGORY_TYPES = {
    column: pd.CategoricalDtype(
        categories=daily_identifiers[column].cat.categories,
        ordered=False,
    )
    for column in CATEGORICAL_FEATURES
}

target_price_source = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    dtype={
        "store_id": "category",
        "item_id": "category",
        "wm_yr_wk": "int32",
        "sell_price": "float32",
    },
)
observed_prices = target_price_source["sell_price"].dropna().to_numpy()
if not np.isfinite(observed_prices).all() or (observed_prices <= 0).any():
    raise ValueError("Некорректные наблюдаемые цены.")
target_week_prices = (
    target_price_source.pivot(index=M5_KEYS, columns="wm_yr_wk", values="sell_price")
    .reindex(index=wrmsse_evaluator.series_index)
    .astype("float32")
)
del target_price_source, observed_prices
gc.collect()


def prepare_categories(frame):
    for column, dtype in CATEGORY_TYPES.items():
        frame[column] = frame[column].astype(dtype)
        if frame[column].isna().any():
            raise ValueError(f"Неизвестная категория в {column}.")
    return frame


def relative_price_change(numerator, denominator):
    result = np.full(len(numerator), np.nan, dtype=np.float32)
    valid = np.isfinite(numerator) & np.isfinite(denominator) & (denominator > 0)
    np.divide(numerator, denominator, out=result, where=valid)
    return result - np.float32(1)


@lru_cache(maxsize=64)
def historical_price_mean_13w(forecast_origin):
    week = int(daily_calendar.loc[forecast_origin, "wm_yr_wk"])
    position = price_week_table.columns.get_loc(week)
    if position < 12:
        raise ValueError("Для среднего цены нужны 13 недель истории.")
    return (
        price_week_table.iloc[:, position - 12 : position + 1]
        .mean(axis=1)
        .to_numpy(dtype=np.float32)
    )


def make_daily_features_final(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    frame = make_daily_features_with_cross_store(forecast_origin, target_date)
    if frame.columns.tolist() != BASE_FEATURE_COLUMNS or not frame.index.equals(
        wrmsse_evaluator.series_index
    ):
        raise ValueError("Изменились базовые признаки или порядок рядов.")
    target_week = int(daily_calendar.loc[target_date, "wm_yr_wk"])
    target_price = target_week_prices[target_week].to_numpy()
    last_price = frame["price_last_known"].to_numpy()
    frame["price_target_week"] = target_price
    frame["price_target_vs_last"] = relative_price_change(target_price, last_price)
    frame["price_target_vs_mean_13w"] = relative_price_change(
        target_price, historical_price_mean_13w(forecast_origin)
    )
    frame["price_target_missing"] = (~np.isfinite(target_price)).astype("float32")
    return prepare_categories(frame.loc[:, FEATURE_COLUMNS])

In [14]:
def make_training_plan(label_cutoff, horizons_per_origin=16):
    if horizons_per_origin not in (2, 4, 8, 16, 28):
        raise ValueError("Допустимы планы 52 × 2, 4, 8, 16 или 28.")
    label_cutoff = pd.Timestamp(label_cutoff)
    origins = pd.date_range(
        end=label_cutoff - pd.Timedelta(days=HORIZON),
        periods=N_ORIGINS,
        freq="7D",
    )
    rows = []
    for number, origin in enumerate(origins):
        first = 5 * number % HORIZON + 1
        old = {first, (5 * number + HORIZON // 2) % HORIZON + 1}
        if number == N_ORIGINS - 1:
            old = {1, HORIZON}
        four = old | (
            {10, 19}
            if number == N_ORIGINS - 1
            else {(first - 1 + shift) % HORIZON + 1 for shift in (7, 21)}
        )
        eight = four | (
            {5, 14, 23, 26}
            if number == N_ORIGINS - 1
            else {(first - 1 + shift) % HORIZON + 1 for shift in (3, 10, 17, 24)}
        )
        if len(old) != 2 or len(four) != 4 or len(eight) != 8:
            raise ValueError("Изменился исходный план горизонтов.")
        horizons = set({2: old, 4: four}.get(horizons_per_origin, eight))
        while len(horizons) < horizons_per_origin:
            candidates = set(range(1, HORIZON + 1)) - horizons

            def separation(candidate):
                return min(
                    min(abs(candidate - old), HORIZON - abs(candidate - old)) for old in horizons
                )

            selected = max(
                candidates,
                key=lambda candidate: (separation(candidate), -candidate),
            )
            horizons.add(selected)
        for horizon in sorted(horizons):
            target_date = origin + pd.Timedelta(days=horizon)
            if target_date > label_cutoff:
                raise ValueError("Цель выходит за конец обучающей истории.")
            rows.append(
                {
                    "origin": origin,
                    "horizon": horizon,
                    "target_date": target_date,
                    "present_in_52x2": horizon in old,
                    "present_in_52x4": horizon in four,
                    "present_in_52x8": horizon in eight,
                }
            )
    return pd.DataFrame(rows)


def clear_feature_caches():
    for function in (
        price_features_at_origin,
        regularity_features_at_origin,
        group_statistics_at_origin,
        cross_store_statistics_at_origin,
        historical_price_mean_13w,
    ):
        function.cache_clear()
    gc.collect()


def build_feature_dataset(pairs, label_cutoff=None):
    pairs = list(pairs)
    if not pairs:
        raise ValueError("Пустой план подготовки данных.")
    n_series = len(wrmsse_evaluator.series_index)
    n_rows = len(pairs) * n_series
    storage = None
    y = np.empty(n_rows, dtype=np.float32) if label_cutoff is not None else None
    if label_cutoff is not None:
        label_cutoff = pd.Timestamp(label_cutoff)
    for number, (origin, target_date) in enumerate(pairs, start=1):
        origin = pd.Timestamp(origin)
        target_date = pd.Timestamp(target_date)
        frame = make_daily_features_final(origin, target_date)
        if storage is None:
            storage = {
                column: np.empty(
                    n_rows,
                    dtype=(
                        frame[column].cat.codes.dtype
                        if column in CATEGORICAL_FEATURES
                        else frame[column].dtype
                    ),
                )
                for column in FEATURE_COLUMNS
            }
        start = (number - 1) * n_series
        stop = start + n_series
        for column in FEATURE_COLUMNS:
            values = (
                frame[column].cat.codes.to_numpy()
                if column in CATEGORICAL_FEATURES
                else frame[column].to_numpy()
            )
            storage[column][start:stop] = values
        if y is not None:
            position = daily_history_dates.get_indexer([target_date])[0]
            if (
                position < 0
                or target_date > label_cutoff
                or origin >= target_date
                or origin > label_cutoff
            ):
                raise ValueError("Недоступная дата обучающей цели.")
            y[start:stop] = daily_history_values[:, position]
            if number % max(1, len(pairs) // 4) == 0 or number == len(pairs):
                print(f"Подготовлено пар дат: {number}/{len(pairs)}", flush=True)
    if y is not None and (not np.isfinite(y).all() or (y < 0).any()):
        raise ValueError("Некорректные целевые продажи.")
    data = {
        column: (
            pd.Categorical.from_codes(storage[column], dtype=CATEGORY_TYPES[column])
            if column in CATEGORICAL_FEATURES
            else storage[column]
        )
        for column in FEATURE_COLUMNS
    }
    return pd.DataFrame(data, columns=FEATURE_COLUMNS, copy=False), y

In [15]:
ACTIVE_PERIOD = None
ACTIVE_ORIGIN = None
_accepted_daily_features = make_daily_features_final


def make_daily_features_final(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    if ACTIVE_ORIGIN is None or forecast_origin > ACTIVE_ORIGIN:
        raise ValueError("Дата отсчёта выходит за разрешённую историю периода.")
    return _accepted_daily_features(forecast_origin, target_date)


def forecast_dates(period):
    origin = PERIOD_SPECS[period]["origin"]
    return pd.date_range(origin + pd.Timedelta(days=1), periods=HORIZON, freq="D", name="date")


def activate_period(period):
    global ACTIVE_PERIOD, ACTIVE_ORIGIN, daily_history, daily_history_dates, daily_history_values
    global group_daily_sales, cross_store_daily_totals
    clear_feature_caches()
    ACTIVE_PERIOD = period
    ACTIVE_ORIGIN = PERIOD_SPECS[period]["origin"]
    daily_history = train_daily.loc[:, :ACTIVE_ORIGIN]
    daily_history_dates = daily_history.columns
    daily_history_values = daily_history.to_numpy(copy=False)
    if daily_history_dates[-1] != ACTIVE_ORIGIN:
        raise ValueError("Неверная граница истории.")
    group_daily_sales = np.asarray(group_aggregation @ daily_history_values, dtype=np.float64)
    cross_store_daily_totals = np.asarray(
        cross_store_aggregation @ daily_history_values, dtype=np.float64
    )
    print(
        f"{period}: история до {ACTIVE_ORIGIN.date()}, {daily_history.shape[1]} дней.", flush=True
    )


def model_training_plan(period, spec):
    origin = PERIOD_SPECS[period]["origin"]
    plan = make_training_plan(origin, spec["density"])
    if spec["week"] is not None:
        first = 7 * (spec["week"] - 1) + 1
        plan = plan.loc[plan["horizon"].between(first, first + 6)].reset_index(drop=True)
        expected_pairs = N_ORIGINS * 7
    else:
        expected_pairs = N_ORIGINS * spec["density"]
    if (
        len(plan) != expected_pairs
        or plan["origin"].nunique() != N_ORIGINS
        or (plan["target_date"] > origin).any()
        or (plan["origin"] >= plan["target_date"]).any()
    ):
        raise ValueError("Неверный план обучения.")
    return plan


def model_prediction_horizons(spec):
    name = spec["name"]
    return [
        horizon
        for horizon in range(1, HORIZON + 1)
        if WEEK_WEIGHTS.loc[(horizon - 1) // 7 + 1, name] > 0
    ]


def plan_csv_bytes(plan):
    return plan.to_csv(index=False, date_format="%Y-%m-%d").encode()


preflight_rows = []
for period in PERIODS_TO_RUN:
    activate_period(period)
    dates = forecast_dates(period)
    if not dates.isin(daily_calendar.index).all():
        raise ValueError("В календаре нет будущих дат.")
    for spec in MODEL_SPECS:
        plan = model_training_plan(period, spec)
        preflight_rows.append(
            {
                "period": period,
                "model": spec["name"],
                "trees": spec["trees"],
                "train_rows": len(plan) * len(series_index),
                "label_end": str(plan["target_date"].max().date()),
                "predict_days": len(model_prediction_horizons(spec)),
            }
        )
    for horizon in (1, 28):
        probe = make_daily_features_final(ACTIVE_ORIGIN, dates[horizon - 1])
        if probe.columns.tolist() != FEATURE_COLUMNS or not probe.index.equals(series_index):
            raise ValueError("Изменился состав признаков или порядок рядов.")
        for column in FEATURE_COLUMNS:
            if column not in CATEGORICAL_FEATURES and np.isinf(probe[column].to_numpy()).any():
                raise ValueError(f"Бесконечность в признаке {column}.")
    del probe
    clear_feature_caches()
display(pd.DataFrame(preflight_rows))

public: история до 2016-04-24, 1913 дней.
private: история до 2016-05-22, 1941 дней.


,period,model,trees,train_rows,label_end,predict_days
0,public,global_trial9_28,197,44393440,2016-04-24,14
1,public,global_trial5_8,471,12683840,2016-04-24,14
2,public,week_01,2076,11098360,2016-04-03,7
3,public,week_02,196,11098360,2016-04-10,7
4,public,week_03,216,11098360,2016-04-17,7
5,public,week_04,93,11098360,2016-04-24,7
6,private,global_trial9_28,197,44393440,2016-05-22,14
7,private,global_trial5_8,471,12683840,2016-05-22,14
8,private,week_01,2076,11098360,2016-05-01,7
9,private,week_02,196,11098360,2016-05-08,7


## 6. Сохранение и продолжение

После каждого обученного компонента сохраняются модель, её протокол и checkpoint.
После получения прогноза сохраняется и таблица предсказаний. Завершённый компонент
повторно не обучается. Если модель уже сохранена, а прогноз ещё не готов, повторяется
только построение её прогноза.

Чтобы продолжить в новой Kaggle-сессии, подключить **`demand-planner_final_checkpoint.zip`**
как Input и снова выполнить ноутбук. Поддерживаются ZIP, распакованная папка и Output
прежнего запуска. Нужны также исходные файлы M5. При нескольких копиях указать одну
в `RESUME_FROM`. Готовые данные сверяются по хешам и параметрам.

При изменении рецепта или среды нужен другой `RUN_NAME`. Файл блокировки не надо
удалять вручную: он защищает папку от двух одновременных обучений.

In [16]:
CONTRACT = {
    "schema": 1,
    "run_name": RUN_NAME,
    "recipe": FROZEN_RECIPE,
    "periods": {
        name: {
            "origin": str(spec["origin"].date()),
            "suffix": spec["suffix"],
            "last_day": spec["last_day"],
        }
        for name, spec in PERIOD_SPECS.items()
    },
    "n_origins": N_ORIGINS,
    "horizon": HORIZON,
    "num_threads": NUM_THREADS,
    "versions": VERSIONS,
    "raw_sha256": RAW_SHA256,
    "compute_use_bottleneck": False,
    "public_labels_used_for_training_or_selection": False,
}
CONTRACT_HASH = object_hash(CONTRACT)


def allowed_checkpoint_path(relative):
    relative = Path(relative)
    return (
        not relative.is_absolute()
        and ".." not in relative.parts
        and any(
            relative.is_relative_to(prefix)
            for prefix in (
                RESULT_DIR.relative_to(PROJECT_ROOT),
                MODEL_DIR.relative_to(PROJECT_ROOT),
            )
        )
    )


def restore_final_checkpoint(source):
    source = Path(source)
    manifest_relative = MANIFEST_PATH.relative_to(PROJECT_ROOT)
    if source.is_file():
        if not is_zipfile(source):
            raise ValueError("Нужен ZIP или распакованная папка final checkpoint.")
        with ZipFile(source) as archive:
            saved = json.loads(archive.read(manifest_relative.as_posix()))
            if saved.get("contract_sha256") != CONTRACT_HASH or saved.get("contract") != CONTRACT:
                raise RuntimeError("Checkpoint создан с другими данными, настройками или версиями.")
            for info in archive.infolist():
                if info.is_dir():
                    continue
                relative = Path(info.filename)
                if (
                    not allowed_checkpoint_path(relative)
                    or (info.external_attr >> 16) & 0o170000 == 0o120000
                ):
                    raise ValueError(f"Недопустимый файл checkpoint: {relative}")
                target = PROJECT_ROOT / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
    elif source.is_dir():
        saved = json.loads((source / manifest_relative).read_text())
        if saved.get("contract_sha256") != CONTRACT_HASH or saved.get("contract") != CONTRACT:
            raise RuntimeError("Checkpoint создан с другими данными, настройками или версиями.")
        for prefix in (RESULT_DIR.relative_to(PROJECT_ROOT), MODEL_DIR.relative_to(PROJECT_ROOT)):
            for incoming in sorted((source / prefix).rglob("*")):
                if incoming.is_symlink():
                    raise ValueError("Ссылки в checkpoint не поддерживаются.")
                if incoming.is_file():
                    target = PROJECT_ROOT / incoming.relative_to(source)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(incoming, target)
    else:
        raise FileNotFoundError(source)
    print("Восстановлено из:", source)


def _export_final_checkpoint_unlocked():
    rows = []
    for period in PERIOD_SPECS:
        for spec in MODEL_SPECS:
            directory = RESULT_DIR / period / "components" / spec["name"]
            fit_path, complete_path = directory / "fit.json", directory / "protocol.json"
            if fit_path.is_file():
                fit = json.loads(fit_path.read_text())
                rows.append(
                    {
                        "period": period,
                        "model": spec["name"],
                        "trees": fit["trees"],
                        "fit_seconds": fit["fit_seconds"],
                        "forecast_saved": complete_path.is_file(),
                    }
                )
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(
        rows, columns=["period", "model", "trees", "fit_seconds", "forecast_saved"]
    ).to_csv(RESULT_DIR / "component_status.csv", index=False)
    atomic_json(
        RESULT_DIR / "run_status.json",
        {
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            "fitted_models": len(rows),
            "completed_forecasts": sum(row["forecast_saved"] for row in rows),
            "total_models_for_both_periods": len(MODEL_SPECS) * 2,
        },
    )
    temporary = CHECKPOINT_PATH.with_suffix(".tmp.zip")
    with ZipFile(temporary, "w", ZIP_DEFLATED, compresslevel=1) as archive:
        for directory in (RESULT_DIR, MODEL_DIR):
            for path in sorted(directory.rglob("*")):
                if (
                    path.is_file()
                    and not path.name.endswith((".tmp", ".lock"))
                    and ".tmp." not in path.name
                ):
                    archive.write(path, path.relative_to(PROJECT_ROOT))
    temporary.replace(CHECKPOINT_PATH)
    print(
        f"Checkpoint: {sum(row['forecast_saved'] for row in rows)}/12 прогнозов готово.", flush=True
    )


def export_final_checkpoint():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    with (MODEL_DIR / "checkpoint.lock").open("a+") as archive_lock:
        fcntl.flock(archive_lock, fcntl.LOCK_EX)
        _export_final_checkpoint_unlocked()


if not MANIFEST_PATH.exists():
    source = RESUME_FROM
    if source is None:
        archives = sorted(
            path for path in Path("/kaggle/input").rglob(CHECKPOINT_PATH.name) if path.is_file()
        )
        suffix = MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix()
        folders = sorted(
            {
                path.parents[3]
                for path in Path("/kaggle/input").rglob("workspace_manifest.json")
                if path.is_file() and path.as_posix().endswith("/" + suffix)
            }
        )
        sources = [*archives, *folders]
        if len(sources) > 1:
            raise ValueError(f"Несколько final checkpoint. Укажи RESUME_FROM: {sources}")
        if sources:
            source = sources[0]
    if source is not None:
        restore_final_checkpoint(source)
if MANIFEST_PATH.exists():
    saved = json.loads(MANIFEST_PATH.read_text())
    if saved.get("contract_sha256") != CONTRACT_HASH or saved.get("contract") != CONTRACT:
        raise RuntimeError("Изменились данные или настройки. Задай другой RUN_NAME.")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
atomic_json(MANIFEST_PATH, {"contract_sha256": CONTRACT_HASH, "contract": CONTRACT})
atomic_json(RESULT_DIR / "frozen_recipe.json", FROZEN_RECIPE)
WEEK_WEIGHTS.to_csv(RESULT_DIR / "frozen_week_weights.csv")
pd.DataFrame(preflight_rows).to_csv(RESULT_DIR / "training_jobs.csv", index=False)
export_final_checkpoint()

Checkpoint: 0/12 прогнозов готово.


In [17]:
def component_paths(period, spec):
    directory = RESULT_DIR / period / "components" / spec["name"]
    model_path = MODEL_DIR / period / f"{spec['name']}.txt"
    return directory, model_path


def job_contract(period, spec):
    plan = model_training_plan(period, spec)
    return {
        "workspace_sha256": CONTRACT_HASH,
        "period": period,
        "origin": str(PERIOD_SPECS[period]["origin"].date()),
        "model": spec,
        "training_plan_sha256": hashlib.sha256(plan_csv_bytes(plan)).hexdigest(),
        "prediction_horizons": model_prediction_horizons(spec),
    }


def component_dates(period, spec):
    dates = forecast_dates(period)
    return dates[np.array(model_prediction_horizons(spec)) - 1]


def read_component_frame(period, spec, protocol):
    directory, _ = component_paths(period, spec)
    _, payload = resolve_prediction_payload(directory, protocol["prediction_payload_sha256"])
    prediction = pd.read_csv(
        io.BytesIO(payload),
        index_col=M5_KEYS,
        dtype={key: "string" for key in M5_KEYS},
        float_precision="round_trip",
    )
    prediction.columns = pd.DatetimeIndex(pd.to_datetime(prediction.columns), name="date")
    if (
        prediction.index.has_duplicates
        or not prediction.index.equals(series_index)
        or not prediction.columns.equals(component_dates(period, spec))
    ):
        raise ValueError("Изменились товары, магазины или даты компонента.")
    values = prediction.to_numpy()
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Некорректные сохранённые прогнозы.")
    return prediction


def load_saved_fit(period, spec):
    directory, model_path = component_paths(period, spec)
    path = directory / "fit.json"
    if not path.is_file():
        return None
    fit = json.loads(path.read_text())
    if fit.get("job") != job_contract(period, spec):
        raise RuntimeError(f"Изменились настройки {period}/{spec['name']}.")
    if not model_path.is_file() or sha256_file(model_path) != fit["model_sha256"]:
        raise RuntimeError(f"Модель отсутствует или изменилась: {model_path}")
    plan_path = directory / "training_plan.csv"
    if not plan_path.is_file() or sha256_file(plan_path) != fit["job"]["training_plan_sha256"]:
        raise RuntimeError("Изменился сохранённый план обучения.")
    if fit["trees"] != spec["trees"]:
        raise RuntimeError("Изменилось число деревьев.")
    return fit


def load_complete_component(period, spec):
    directory, _ = component_paths(period, spec)
    path = directory / "protocol.json"
    if not path.is_file():
        return None
    fit = load_saved_fit(period, spec)
    if fit is None:
        raise RuntimeError("Есть прогноз без протокола обучения.")
    protocol = json.loads(path.read_text())
    if protocol.get("job") != fit["job"] or protocol.get("model_sha256") != fit["model_sha256"]:
        raise RuntimeError("Прогноз и модель относятся к разным запускам.")
    return protocol, read_component_frame(period, spec, protocol)


def check_feature_table(frame):
    if frame.columns.tolist() != FEATURE_COLUMNS:
        raise ValueError("Изменился состав признаков.")
    for column in CATEGORICAL_FEATURES:
        if frame[column].dtype != CATEGORY_TYPES[column] or frame[column].isna().any():
            raise ValueError(f"Неверные категории: {column}.")
    for column in FEATURE_COLUMNS:
        if column not in CATEGORICAL_FEATURES and np.isinf(frame[column].to_numpy()).any():
            raise ValueError(f"Бесконечность в {column}.")


def predict_component(period, spec, booster):
    if booster.feature_name() != FEATURE_COLUMNS or booster.current_iteration() != spec["trees"]:
        raise RuntimeError("Состав признаков или число деревьев модели изменились.")
    if ACTIVE_PERIOD != period:
        activate_period(period)
    dates = component_dates(period, spec)
    output = np.empty((len(series_index), len(dates)), dtype=np.float64)
    for position, day in enumerate(dates):
        frame = make_daily_features_final(PERIOD_SPECS[period]["origin"], day)
        check_feature_table(frame)
        if not frame.index.equals(series_index):
            raise ValueError("Изменился порядок рядов перед predict.")
        output[:, position] = booster.predict(
            frame, num_iteration=spec["trees"], num_threads=NUM_THREADS
        )
    if not np.isfinite(output).all() or (output < 0).any():
        raise ValueError("Некорректный прогноз модели.")
    return pd.DataFrame(output, index=series_index, columns=dates)

## 7. Обучение выбранных моделей

Последовательно обучаем шесть компонентов каждого периода. На одной машине
одновременно хранится обучающая таблица только текущей модели. После бинирования
исходная таблица освобождается. Общие модели прогнозируют только первую и третью
недели: на остальных двух их вес равен нулю.

Все деревья используются в заранее выбранном количестве. Ни public-метрика,
ни public-факты не передаются в обучение или раннюю остановку.

In [18]:
class FinalTimeBudgetExceeded(Exception):
    pass


def final_deadline_callback(deadline):
    def check(env):
        if perf_counter() >= deadline:
            raise FinalTimeBudgetExceeded("Закончился бюджет времени запуска.")

    check.order = 25
    check.before_iteration = False
    return check


def fit_component(period, spec, deadline):
    clear_feature_caches()
    plan = model_training_plan(period, spec)
    pairs = plan[["origin", "target_date"]].itertuples(index=False, name=None)
    origin = PERIOD_SPECS[period]["origin"]
    if ACTIVE_PERIOD != period:
        activate_period(period)
    print(
        f"Подготовка {period}/{spec['name']}: {len(plan) * len(series_index):,} строк.", flush=True
    )
    X_train, y_train = build_feature_dataset(pairs, label_cutoff=origin)
    check_feature_table(X_train)
    if len(X_train) != len(plan) * len(series_index):
        raise ValueError("Неверное число обучающих строк.")
    table_gib = float(X_train.memory_usage(index=True, deep=True).sum() / 1024**3)
    print(f"Train: {table_gib:.2f} GiB. Бинирование.", flush=True)
    dataset_params = {
        key: spec["params"][key]
        for key in (
            "max_bin",
            "bin_construct_sample_cnt",
            "feature_pre_filter",
            "seed",
            "verbosity",
        )
    }
    dataset_params["num_threads"] = NUM_THREADS
    train_set = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=CATEGORICAL_FEATURES,
        params=dataset_params,
        free_raw_data=True,
    )
    train_set.construct()
    del X_train, y_train
    clear_feature_caches()
    gc.collect()
    if perf_counter() >= deadline:
        del train_set
        raise FinalTimeBudgetExceeded("Бюджет закончился после подготовки данных.")
    started = perf_counter()

    def progress(env):
        iteration = env.iteration + 1
        if iteration % 100 == 0 or iteration == spec["trees"]:
            print(
                f"{period}/{spec['name']}: {iteration}/{spec['trees']} деревьев, "
                f"{(perf_counter() - started) / 60:.1f} мин.",
                flush=True,
            )

    progress.order = 22
    params = {**spec["params"], "num_threads": NUM_THREADS}
    try:
        booster = lgb.train(
            params,
            train_set,
            num_boost_round=spec["trees"],
            callbacks=[progress, final_deadline_callback(deadline)],
        )
    finally:
        del train_set
        gc.collect()
    if booster.current_iteration() != spec["trees"]:
        raise RuntimeError("Модель не обучила заданное число деревьев.")
    directory, model_path = component_paths(period, spec)
    directory.mkdir(parents=True, exist_ok=True)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = model_path.with_suffix(".tmp.txt")
    booster.save_model(str(temporary), num_iteration=spec["trees"])
    temporary.replace(model_path)
    (directory / "training_plan.csv").write_bytes(plan_csv_bytes(plan))
    fit = {
        "job": job_contract(period, spec),
        "trees": int(booster.current_iteration()),
        "model_sha256": sha256_file(model_path),
        "fit_seconds": float(perf_counter() - started),
        "train_rows": len(plan) * len(series_index),
        "raw_table_gib": table_gib,
        "max_label_date": str(plan["target_date"].max().date()),
        "early_stopping": False,
        "fitted_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    pd.Series(
        booster.feature_importance(importance_type="gain"),
        index=FEATURE_COLUMNS,
        name="importance_gain",
    ).sort_values(ascending=False).to_csv(directory / "feature_importance_gain.csv")
    atomic_json(directory / "fit.json", fit)
    export_final_checkpoint()
    return booster, fit


def run_final_models():
    with (MODEL_DIR / "training.lock").open("a+") as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError("В этой папке уже работает другое обучение.") from error
        started = perf_counter()
        deadline = started + TIMEOUT_HOURS * 3600
        status = "finished_requested_periods"
        booster = None
        try:
            total = len(PERIODS_TO_RUN) * len(MODEL_SPECS)
            job_number = 0
            for period in PERIODS_TO_RUN:
                activate_period(period)
                for spec in MODEL_SPECS:
                    job_number += 1
                    label = f"{period}/{spec['name']}"
                    if load_complete_component(period, spec) is not None:
                        print(f"{job_number}/{total}: {label} — готово, пропускаем.", flush=True)
                        continue
                    if perf_counter() >= deadline:
                        raise FinalTimeBudgetExceeded(
                            "Бюджет закончился перед следующим компонентом."
                        )
                    print(f"{job_number}/{total}: {label}.", flush=True)
                    fit = load_saved_fit(period, spec)
                    if fit is None:
                        booster, fit = fit_component(period, spec, deadline)
                    else:
                        _, model_path = component_paths(period, spec)
                        booster = lgb.Booster(model_file=str(model_path))
                        print("Модель уже сохранена: строим только прогноз.", flush=True)
                    prediction = predict_component(period, spec, booster)
                    directory, _ = component_paths(period, spec)
                    path = directory / "predictions.csv.gz"
                    temporary = directory / "predictions.tmp.csv.gz"
                    prediction.to_csv(temporary, compression="gzip")
                    temporary.replace(path)
                    protocol = {
                        "job": fit["job"],
                        "model_sha256": fit["model_sha256"],
                        "prediction_payload_sha256": hashlib.sha256(
                            read_csv_payload(path)
                        ).hexdigest(),
                        "forecast_dates": [str(day.date()) for day in prediction.columns],
                        "created_at_utc": datetime.now(timezone.utc).isoformat(),
                    }
                    atomic_json(directory / "protocol.json", protocol)
                    booster = None
                    del prediction
                    clear_feature_caches()
                    gc.collect()
                    export_final_checkpoint()
        except FinalTimeBudgetExceeded as error:
            status = "time_budget"
            print(str(error), "Checkpoint сохранён; продолжи в новой сессии.", flush=True)
        except BaseException:
            status = "exception_or_interrupt"
            raise
        finally:
            booster = None
            clear_feature_caches()
            gc.collect()
            atomic_json(
                RESULT_DIR / "invocation_status.json",
                {
                    "status": status,
                    "elapsed_seconds": float(perf_counter() - started),
                    "budget_hours": TIMEOUT_HOURS,
                    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
                },
            )
            export_final_checkpoint()


if RUN_TRAINING:
    run_final_models()
else:
    print("RUN_TRAINING=False: читаем только уже готовые результаты.")

public: история до 2016-04-24, 1913 дней.
1/12: public/global_trial9_28.
Подготовка public/global_trial9_28: 44,393,440 строк.
Подготовлено пар дат: 364/1456
Подготовлено пар дат: 728/1456
[Промежуточный журнал обучения сокращён; метрики и графики сохранены.]
Train: 9.76 GiB. Бинирование.
public/global_trial9_28: 100/197 деревьев, 12.2 мин.
public/global_trial9_28: 197/197 деревьев, 21.7 мин.
Checkpoint: 0/12 прогнозов готово.
Checkpoint: 1/12 прогнозов готово.
2/12: public/global_trial5_8.
Подготовка public/global_trial5_8: 12,683,840 строк.
Train: 2.79 GiB. Бинирование.
public/global_trial5_8: 100/471 деревьев, 4.3 мин.
public/global_trial5_8: 200/471 деревьев, 7.2 мин.
public/global_trial5_8: 300/471 деревьев, 10.2 мин.
public/global_trial5_8: 400/471 деревьев, 13.3 мин.
public/global_trial5_8: 471/471 деревьев, 15.4 мин.
Checkpoint: 1/12 прогнозов готово.
Checkpoint: 2/12 прогнозов готово.
3/12: public/week_01.
Подготовка public/week_01: 11,098,360 строк.
Train: 2.44 GiB. Бинирован

## 8. Сборка прогноза

Для каждой недели применяем сохранённые веса, затем один раз умножаем результат
на **1.006**. Никакого нового поиска весов здесь нет. Прогнозы остаются дробными:
округление до целых не требуется для submission и меняло бы выбранный рецепт.

In [19]:
def compose_frozen_forecast(period, component_frames):
    dates = forecast_dates(period)
    output = np.zeros((len(series_index), HORIZON), dtype=np.float64)
    for position, day in enumerate(dates):
        week = position // 7 + 1
        for spec in MODEL_SPECS:
            weight = float(WEEK_WEIGHTS.loc[week, spec["name"]])
            if weight == 0:
                continue
            frame = component_frames.get(spec["name"])
            if frame is None or not frame.index.equals(series_index) or day not in frame.columns:
                raise ValueError(f"Нет нужного прогноза {period}/{spec['name']} на {day}.")
            output[:, position] += weight * frame[day].to_numpy(dtype=np.float64)
    uncalibrated = pd.DataFrame(output, index=series_index, columns=dates)
    calibrated = uncalibrated * CALIBRATION_COEFFICIENT
    if not np.isfinite(calibrated.to_numpy()).all() or (calibrated.to_numpy() < 0).any():
        raise ValueError("Некорректный итоговый прогноз.")
    return uncalibrated, calibrated


FINAL_FORECASTS = {}
UNCALIBRATED_FORECASTS = {}
for period in PERIOD_SPECS:
    component_frames = {}
    missing = []
    for spec in MODEL_SPECS:
        loaded = load_complete_component(period, spec)
        if loaded is None:
            missing.append(spec["name"])
        else:
            component_frames[spec["name"]] = loaded[1]
    if missing:
        print(f"{period}: пока нет компонентов {missing}.", flush=True)
        continue
    uncalibrated, calibrated = compose_frozen_forecast(period, component_frames)
    UNCALIBRATED_FORECASTS[period] = uncalibrated
    FINAL_FORECASTS[period] = calibrated
    directory = RESULT_DIR / period / "final_forecast"
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / "predictions.csv.gz"
    calibrated.to_csv(path, compression="gzip")
    atomic_json(
        directory / "protocol.json",
        {
            "workspace_sha256": CONTRACT_HASH,
            "period": period,
            "origin": str(PERIOD_SPECS[period]["origin"].date()),
            "forecast_dates": [str(day.date()) for day in calibrated.columns],
            "prediction_payload_sha256": hashlib.sha256(read_csv_payload(path)).hexdigest(),
            "recipe": FROZEN_RECIPE,
            "weights_reselected": False,
            "calibration_applied_once": True,
        },
    )
    print(f"{period}: готов прогноз {calibrated.shape[0]:,} × {calibrated.shape[1]}.", flush=True)
del component_frames
export_final_checkpoint()

public: готов прогноз 30,490 × 28.
private: готов прогноз 30,490 × 28.
Checkpoint: 12/12 прогнозов готово.


## 9. Качество на public-периоде

Теперь можно открыть public-факты из evaluation-файла и посчитать WRMSSE за
**25 апреля — 22 мая**. Веса и масштабы метрики определены по истории до 24 апреля.
Для сравнения оставляем среднее по дням недели за последние четыре недели.

Этот отчёт не меняет число деревьев, веса ансамбля моделей и калибровку. Он показывает перенос
выбранного рецепта. Public-период уже рассматривался в ранних экспериментах проекта,
поэтому это не полностью нетронутый тест. Для private локальных фактов в наборе M5
нет: его score появится только при внешней оценке готового файла.

In [20]:
if "public" in FINAL_FORECASTS:
    public_dates = forecast_dates("public")
    public_actual = train_daily.loc[series_index, public_dates]
    known = train_daily.loc[:, :PUBLIC_ORIGIN].iloc[:, -28:]
    baseline_values = np.column_stack(
        [
            known.loc[:, known.columns.dayofweek == day.dayofweek].mean(axis=1).to_numpy()
            for day in public_dates
        ]
    )
    baseline = pd.DataFrame(baseline_values, index=series_index, columns=public_dates)
    public_rows = []
    for label, prediction in [
        ("Среднее по дням недели за 4 недели", baseline),
        ("Фиксированный ансамбль моделей без калибровки", UNCALIBRATED_FORECASTS["public"]),
        ("Фиксированный ансамбль моделей × 1.006", FINAL_FORECASTS["public"]),
    ]:
        score, levels = wrmsse_evaluator.score(public_actual, prediction)
        public_rows.append(
            {
                "model": label,
                "WRMSSE": float(score),
                "Bias": float(prediction.to_numpy().sum() / public_actual.to_numpy().sum() - 1),
            }
        )
        if label == "Фиксированный ансамбль моделей × 1.006":
            levels.to_csv(RESULT_DIR / "public/final_forecast/metrics_by_level.csv")
    public_metrics = pd.DataFrame(public_rows).set_index("model")
    public_metrics.to_csv(RESULT_DIR / "public_metrics.csv")
    display(public_metrics.style.format({"WRMSSE": "{:.6f}", "Bias": "{:+.2%}"}))
    print("Это локальная оценка public-периода. Рецепт после неё не меняется.")
else:
    print("Public-прогноз ещё не готов: оценку пока не считаем.")

,WRMSSE,Bias
model,,
Среднее по дням недели за 4 недели,0.752421,-3.91%
Фиксированный ансамбль моделей без калибровки,0.627435,-4.99%
Фиксированный ансамбль моделей × 1.006,0.613284,-4.42%


Это локальная оценка public-периода. Рецепт после неё не меняется.


## 10. Файл для Kaggle

Строки `_validation` получают отдельный прогноз public, построенный без public-фактов.
Строки `_evaluation` получают прогноз private после переобучения на истории до d_1941.
Порядок строк берём из `sample_submission.csv`; сопоставление идёт по `id`.

Проверяем количество строк, уникальность id, 28 столбцов прогноза и отсутствие
пропусков, бесконечностей и отрицательных значений. Если не готов хотя бы один
период, неполный submission не создаётся.

In [21]:
def build_submission(sample, forecasts, submission_ids):
    expected_columns = ["id", *[f"F{day}" for day in range(1, HORIZON + 1)]]
    if sample.columns.tolist() != expected_columns or sample["id"].duplicated().any():
        raise ValueError("Неверный шаблон submission.")
    blocks = []
    for period in ("public", "private"):
        if period not in forecasts:
            raise ValueError(f"Не готов {period}-прогноз.")
        frame = forecasts[period]
        if not frame.index.equals(series_index) or not frame.columns.equals(forecast_dates(period)):
            raise ValueError("Неверный порядок рядов или дат финального прогноза.")
        ids = pd.Index(submission_ids[period].astype(str), name="id")
        if len(ids) != len(frame) or ids.has_duplicates:
            raise ValueError("Неверное сопоставление id и рядов.")
        blocks.append(
            pd.DataFrame(frame.to_numpy(dtype=np.float64), index=ids, columns=expected_columns[1:])
        )
    values = pd.concat(blocks)
    template_ids = pd.Index(sample["id"].astype(str), name="id")
    if (
        len(template_ids) != 2 * len(series_index)
        or values.index.has_duplicates
        or len(values) != len(template_ids)
        or not template_ids.isin(values.index).all()
    ):
        raise ValueError("Набор id отличается от sample_submission.")
    result = values.reindex(template_ids).reset_index()
    numbers = result.iloc[:, 1:].to_numpy()
    if not np.isfinite(numbers).all() or (numbers < 0).any():
        raise ValueError("Некорректные прогнозы в submission.")
    if (
        result.columns.tolist() != expected_columns
        or result["id"].tolist() != sample["id"].astype(str).tolist()
    ):
        raise ValueError("Изменился порядок шаблона.")
    return result


if set(FINAL_FORECASTS) == set(PERIOD_SPECS):
    submission = build_submission(sample_submission, FINAL_FORECASTS, SUBMISSION_IDS)
    if submission.shape != (60_980, 29):
        raise ValueError("Полный M5 submission должен иметь размер 60 980 × 29.")
    stored_submission = RESULT_DIR / "submission.csv"
    temporary = stored_submission.with_suffix(".tmp.csv")
    submission.to_csv(temporary, index=False, float_format="%.17g")
    temporary.replace(stored_submission)
    shutil.copy2(stored_submission, SUBMISSION_PATH)
    verification = pd.read_csv(
        SUBMISSION_PATH, dtype={"id": "string"}, float_precision="round_trip"
    )
    if (
        verification.shape != submission.shape
        or verification["id"].astype(str).tolist() != submission["id"].tolist()
        or not np.array_equal(
            verification.iloc[:, 1:].to_numpy(), submission.iloc[:, 1:].to_numpy()
        )
    ):
        raise RuntimeError("Записанный submission не прошёл обратное чтение.")
    atomic_json(
        RESULT_DIR / "submission_manifest.json",
        {
            "workspace_sha256": CONTRACT_HASH,
            "rows": len(submission),
            "columns": submission.columns.tolist(),
            "submission_sha256": sha256_file(SUBMISSION_PATH),
            "calibration_coefficient": CALIBRATION_COEFFICIENT,
            "periods": {
                name: {
                    "origin": str(PERIOD_SPECS[name]["origin"].date()),
                    "forecast_start": str(forecast_dates(name)[0].date()),
                    "forecast_end": str(forecast_dates(name)[-1].date()),
                }
                for name in PERIOD_SPECS
            },
            "weights_reselected_on_public": False,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        },
    )
    display(submission.head())
    print("Готово:", SUBMISSION_PATH, "| Размер:", submission.shape)
    display(FileLink(os.path.relpath(SUBMISSION_PATH, Path.cwd())))
else:
    print(
        "Submission пока не создан. Не готовы периоды:",
        sorted(set(PERIOD_SPECS) - set(FINAL_FORECASTS)),
    )
export_final_checkpoint()
display(FileLink(os.path.relpath(CHECKPOINT_PATH, Path.cwd())))

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,...,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0.871117,0.787317,0.758426,0.787628,0.893735,1.021464,1.057812,0.941775,0.844706,0.802662,0.830141,...,0.798383,0.837831,0.925102,1.234756,1.260497,0.910775,0.818626,0.794890,0.804758,0.921171,1.126018,1.171266
1,HOBBIES_1_002_CA_1_validation,0.162019,0.165890,0.162518,0.168602,0.203803,0.224137,0.251944,0.178422,0.173994,0.170556,0.163241,...,0.178743,0.172938,0.203365,0.260410,0.304320,0.193922,0.256980,0.190271,0.195851,0.211586,0.250103,0.248735
2,HOBBIES_1_003_CA_1_validation,0.390671,0.386073,0.342059,0.364827,0.481296,0.553887,0.593480,0.425585,0.443265,0.381013,0.392361,...,0.408149,0.424597,0.529726,0.609217,0.703216,0.524313,0.463888,0.481460,0.498587,0.542759,0.696474,0.682415
3,HOBBIES_1_004_CA_1_validation,1.866203,1.522640,1.416853,1.561830,1.756489,2.424972,2.546392,2.077702,1.755583,1.610949,1.544048,...,1.466900,1.554499,1.869414,2.598603,2.538385,1.968762,1.615217,1.548598,1.490904,2.045950,2.534588,2.510568
4,HOBBIES_1_005_CA_1_validation,1.265154,1.061227,1.069939,1.029680,1.248054,1.371475,1.459445,1.405815,1.173898,1.110779,1.097868,...,1.049681,1.067053,1.250135,1.585572,1.758179,1.219681,1.075248,1.040458,1.046123,1.243962,1.546385,1.727859


Готово: /kaggle/working/submission.csv | Размер: (60980, 29)


Checkpoint: 12/12 прогнозов готово.


## Результат

Создан `submission.csv`: **60 980 строк × 29 столбцов**. Для public модель
использовала историю до **24 апреля**, для private — до **22 мая 2016**.
Public WRMSSE — **0.613284** против **0.752421** у сезонного прогноза.
Private WRMSSE поздней отправки — **0.58009**.

Итоговые модели, checkpoint и submission опубликованы в
[GitHub Release](https://github.com/dannya220/demand-planner/releases/tag/v1.0.0).
Параметры и метрики находятся в
[results/m5_wrmsse/final_m5_frozen_ensemble_v1](../results/m5_wrmsse/final_m5_frozen_ensemble_v1).

### Ссылки

- [M5 Forecasting — Accuracy: данные и шаблон](https://www.kaggle.com/competitions/m5-forecasting-accuracy/data)
- [LightGBM 4.6.0: обучение с заданным числом итераций](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.train.html)
- [Kaggle Notebooks: сохранение и выполнение](https://www.kaggle.com/docs/notebooks)

Источник выбранного рецепта: сохранённые результаты ноутбуков 05–09. Новые значения
параметров в этом ноутбуке не подбирались.